In [ ]:
import string
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Download necessary NLTK assets
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve the LookupError

# 0. SIMULATED DATASET CREATION

# Replacing missing files with sample SMS data for execution
data = {
    'v1': ['ham', 'spam', 'ham', 'spam', 'ham', 'ham', 'spam', 'ham', 'spam', 'ham'],
    'v2': [
        "Hey, are we still meeting for lunch today?",
        "URGENT! Your mobile number has won a £2,000 prize! Call 09061701461 box4320.",
        "Just wanted to check if you finished the report.",
        "FREE ringtone! Text 'LIVE' to 80082 now to get your copy. T&C apply.",
        "I'll be home a bit late tonight, don't wait up.",
        "Can you please pick up some milk on your way back?",
        "CONGRATULATIONS! You have been selected for a free holiday. Click http://spam.com",
        "Are you free this weekend for a quick catch up?",
        "Private! Your account statement shows un-claimed points. Call 08719523841.",
        "Yeah, that sounds good to me. See you later!"
    ]
}
df = pd.DataFrame(data)
df.columns = ['label', 'message']

# 1. DATASET EXPLORATION

print("   1. DATASET EXPLORATION  ")
# Count spam vs ham messages
label_counts = df['label'].value_counts()
print("Message Counts:\n", label_counts)

# Tokenizer helper for exploration
def quick_tokenize(text):
    return text.lower().split()

# Identify common words in spam messages
spam_words = []
for msg in df[df['label'] == 'spam']['message']:
    spam_words.extend(quick_tokenize(msg))

common_spam = Counter(spam_words).most_common(5)
print("\nMost Common Words in Spam (Raw):", common_spam)
print("\n" + "="*50 + "\n")

# 2. TEXT PREPROCESSING

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercasing
    text = text.lower()

    # Removal of URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Removal of punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    filtered_tokens = [word for word in tokens if word not in stop_words]

    return " ".join(filtered_tokens)

# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess_text)
print("   2. TEXT PREPROCESSING SAMPLE   ")
print(df[['message', 'clean_message']].head(2))
print("\n" + "="*50 + "\n")

# 3. FEATURE ENGINEERING

print("   3. FEATURE ENGINEERING   ")
# Encode target labels: ham = 0, spam = 1
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['clean_message']
y = df['label_encoded']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Convert text into numerical features using TF-IDF Vectorization
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape: {X_test_tfidf.shape}")
print("\n" + "="*50 + "\n")

# 4. MODEL TRAINING

print("   4. MODEL TRAINING   ")
# Train the model using Logistic Regression
model = LogisticRegression(random_state=42)
model.fit(X_train_tfidf, y_train)
print("Logistic Regression model trained successfully.")
print("\n" + "="*50 + "\n")

# 5. EVALUATION

print("   5. EVALUATION   ")
# Predict on test data
y_pred = model.predict(X_test_tfidf)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['ham', 'spam'], zero_division=0))

# Optional Visuals: Plot Confusion Matrix
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
# plt.ylabel('Actual')
# plt.xlabel('Predicted')
# plt.title('Confusion Matrix')
# plt.show()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


   1. DATASET EXPLORATION  
Message Counts:
 label
ham     6
spam    4
Name: count, dtype: int64

Most Common Words in Spam (Raw): [('your', 3), ('a', 2), ('call', 2), ('free', 2), ('to', 2)]


   2. TEXT PREPROCESSING SAMPLE   
                                             message  \
0         Hey, are we still meeting for lunch today?   
1  URGENT! Your mobile number has won a £2,000 pr...   

                                       clean_message  
0                      hey still meeting lunch today  
1  urgent mobile number £2000 prize call 09061701...  


   3. FEATURE ENGINEERING   
Train matrix shape: (7, 42)
Test matrix shape: (3, 42)


   4. MODEL TRAINING   
Logistic Regression model trained successfully.


   5. EVALUATION   
Accuracy:  0.6667
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000

Confusion Matrix:
[[2 0]
 [1 0]]

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.67      1.00      0.80         2
       

## Identifying Spam Messages

Now, let's use the trained model (`model`) and the TF-IDF vectorizer (`tfidf`) to predict whether a new, unseen message is spam or not. The process involves applying the same preprocessing and vectorization steps to the new message as were applied to the training data, and then using the model to make a prediction.

## Code Explanation

This notebook sets up a basic spam message classification system using Logistic Regression. Here's a breakdown of each section:

### 0. SIMULATED DATASET CREATION

-   **Purpose**: Creates a small, in-memory dataset of SMS messages, labeled as either 'ham' (legitimate) or 'spam'. This simulates having a CSV file or database of messages.
-   **Libraries**: Uses `pandas` to create a DataFrame from a dictionary.

### 1. DATASET EXPLORATION

-   **Purpose**: Gets a quick overview of the dataset characteristics.
-   **Key Actions**:
    -   `df['label'].value_counts()`: Shows the distribution of 'ham' vs. 'spam' messages.
    -   `quick_tokenize` function: A simple tokenizer to split messages into words.
    -   `Counter(spam_words).most_common(5)`: Identifies the 5 most frequently occurring words in the raw spam messages, which can give initial insights into spam characteristics.

### 2. TEXT PREPROCESSING

-   **Purpose**: Cleans and normalizes the text data to make it suitable for machine learning.
-   **Key Steps (within `preprocess_text` function)**:
    -   **Lowercasing**: Converts all text to lowercase to treat words like "Free" and "free" as the same.
    -   **URL Removal**: Removes web addresses (e.g., `http://spam.com`) as they often don't contribute to the meaning of the message content.
    -   **Punctuation Removal**: Removes characters like `!`, `?`, `.`, `,` etc., as they are generally not useful for classification.
    -   **Tokenization**: Breaks down sentences into individual words or "tokens" using `nltk.word_tokenize`.
    -   **Stopword Removal**: Removes common English words (e.g., "the", "is", "a") that often carry little semantic value and can add noise to the model.
-   **Libraries**: Uses `re` for regular expressions (URL removal), `string` for punctuation, and `nltk` for tokenization and stopwords.

### 3. FEATURE ENGINEERING

-   **Purpose**: Transforms the preprocessed text into numerical features that a machine learning model can understand.
-   **Key Actions**:
    -   **Label Encoding**: Converts the categorical 'label' (ham/spam) into numerical values (0/1) for model training.
    -   **Data Splitting**: Divides the dataset into training (70%) and testing (30%) sets using `train_test_split`. This is crucial to evaluate the model's performance on unseen data.
    -   **TF-IDF Vectorization**: Converts the `clean_message` text into numerical vectors. TF-IDF (Term Frequency-Inverse Document Frequency) assigns weights to words based on their frequency in a document and their rarity across the entire dataset. This helps highlight important words.
-   **Libraries**: Uses `sklearn.model_selection` for splitting and `sklearn.feature_extraction.text` for TF-IDF.

### 4. MODEL TRAINING

-   **Purpose**: Builds the machine learning model using the preprocessed and vectorized training data.
-   **Model**: Employs `LogisticRegression`, a common algorithm for binary classification tasks (like spam/ham).
-   **Action**: `model.fit(X_train_tfidf, y_train)` trains the model to learn the patterns that differentiate spam from ham based on the TF-IDF features.

### 5. EVALUATION

-   **Purpose**: Assesses how well the trained model performs on the unseen test data.
-   **Key Metrics**:
    -   `accuracy_score`: The proportion of correctly classified messages.
    -   `precision_score`: The proportion of predicted spam messages that were actually spam (minimizes false positives).
    -   `recall_score`: The proportion of actual spam messages that were correctly identified (minimizes false negatives).
    -   `f1_score`: The harmonic mean of precision and recall, providing a single metric that balances both.
    -   `confusion_matrix`: A table showing true positives, true negatives, false positives, and false negatives.
    -   `classification_report`: Provides precision, recall, f1-score, and support for each class.
-   **Libraries**: Uses `sklearn.metrics` for all evaluation metrics.

In [ ]:
# Function to predict if a new message is spam or ham
def predict_spam(message):
    # Preprocess the message using the same function as before
    clean_message = preprocess_text(message)

    # Vectorize the clean message using the *trained* TF-IDF vectorizer
    # We use .transform() here, not .fit_transform(), as it's already fitted
    message_tfidf = tfidf.transform([clean_message]) # Wrap in list as transform expects an iterable

    # Predict the label (0 for ham, 1 for spam)
    prediction = model.predict(message_tfidf)[0]

    # Return the corresponding label string
    return 'spam' if prediction == 1 else 'ham'

# Test with some new messages
new_messages = [
    "Congratulations! You've won a free iPhone. Click here: www.freestuff.com",
    "Hey, are we still on for dinner tonight?",
    "Your account has been compromised. Please verify your details at evil.com",
    "Meeting at 3 PM today. Don't forget."
]

print("\n--- SPAM DETECTION FOR NEW MESSAGES ---")
for msg in new_messages:
    prediction = predict_spam(msg)
    print(f"Message: '{msg}'\nPrediction: {prediction.upper()}\n")



--- SPAM DETECTION FOR NEW MESSAGES ---
Message: 'Congratulations! You've won a free iPhone. Click here: www.freestuff.com'
Prediction: SPAM

Message: 'Hey, are we still on for dinner tonight?'
Prediction: HAM

Message: 'Your account has been compromised. Please verify your details at evil.com'
Prediction: HAM

Message: 'Meeting at 3 PM today. Don't forget.'
Prediction: HAM

